# 11 · Test-time augmentation (TTA) and P2 small-object head

**Owner:** anyone · **GPU:** T4 (TTA evaluation ~minutes; P2 training ~1–1.5 h).
Two follow-up experiments on the final model (`S3_yolov8m_n2000_r1280_full_COCO_s42`), aimed at the weakest result: tiny objects (APt, tiny-object recall 70%).

* **TTA** (no training): the *same* final weights, predicted on several transformed copies of each image and merged.
  * `tta` = Ultralytics built-in TTA (horizontal flip + down-scales 1.0 / 0.83 / 0.67).
  * `tta_up` = flip × up-scales 1.0 / 1.25 (1280 and 1600), merged with class-wise NMS — up-scaling is the variant expected to help tiny objects.
* **P2 head** (one training run): `yolov8m-p2` = YOLOv8m plus an extra detection layer on the stride-4 feature map, initialised from the same COCO `yolov8m.pt` weights; everything else identical to the final model (2000 frames, 1280, same epochs/seed).

Results are written as normal experiment records (stage `S7`), so they show up in `master_results.csv` / notebook 10. Nothing here changes `decisions.json`.
Run `SMOKE_TEST = True` first if you want a quick end-to-end check (smoke records are kept separately).

In [1]:
# ===== 0. Setup — identical in every notebook. Runtime > Change runtime type > T4 GPU =====
SMOKE_TEST = False # True: 16 train / 8 val images, 1 epoch — checks the whole pipeline in minutes
DRIVE_ROOT = "/content/drive/MyDrive/MAVREC" # shared team folder (each member: "Add shortcut to Drive")

import os, sys, subprocess, importlib
try:
    from google.colab import drive
    try:
        os.listdir("/content/drive/MyDrive") # raises if the Drive FUSE mount went stale
        drive.mount("/content/drive")
    except OSError: # "Transport endpoint is not connected" after a Drive hiccup: remount
        drive.mount("/content/drive", force_remount=True)
except ImportError:
    print("Not in Colab — make sure DRIVE_ROOT exists locally.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.173", "transformers==5.18.0", "pycocotools>=2.0.8", "timm>=1.0.15", "tabulate", "tensorboard", "seaborn"], check=True)

CODE_DIR = os.path.join(DRIVE_ROOT, "Final_Project_Code")
assert os.path.exists(os.path.join(CODE_DIR, "mavrec_lib.py")), "Run 00_Setup_and_Data_Preparation first (it installs mavrec_lib.py)."
sys.path.insert(0, CODE_DIR)
import mavrec_lib as ml; importlib.reload(ml)
CFG = ml.setup(DRIVE_ROOT, smoke=SMOKE_TEST or None)
import copy, numpy as np, pandas as pd
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
N_SMALL, N_SCREEN, N_LARGE = ml.TRAIN_SIZES # 500, 1000, 2000

Mounted at /content/drive
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Environment:
  python       : 3.13.15
  torch        : 2.11.0+cu130
  cuda         : True
  torchvision  : 0.26.0+cu130
  ultralytics  : 8.4.173
  transformers : 5.18.0
  pycocotools  : 2.0.11
  numpy        : 2.1.3
  lib          : 1.0.0
  gpu          : Tesla T4
  gpu_mem_gb   : 14.6


## 1 · Register `yolov8m-p2` with the library
`mavrec_lib` only knows the models in its tables, so P2 is added here (copying the YOLOv8m entries: same epochs, batch, model family).
Re-run this cell after any `importlib.reload(ml)`.

In [2]:
P2 = "yolov8m-p2"
def register_p2():
    for table in (ml.MODEL_FAMILY, ml.MODEL_GROUP, ml.MODEL_BACKBONE, ml.PRETRAINED_SOURCE, ml.DEFAULT_EPOCHS, ml.T4_BATCH):
        table[P2] = copy.deepcopy(table["yolov8m"])
    ml.MODEL_BACKBONE[P2] = "CSP-C2f (m) + P2 head"
    ml.PRETRAINED_SOURCE[P2] = "yolov8m.pt (COCO), shape-matched into yolov8m-p2.yaml"
register_p2()

# COCO-initialised P2 checkpoint: build yolov8m-p2 from its yaml and copy in every yolov8m.pt layer whose shape matches
# (backbone + neck + the 3 existing heads; the new P2 branch starts from scratch). Kept on Drive so resumes reuse it.
P2_INIT = os.path.join(DRIVE_ROOT, "Final_Project_Results", "p2_init", "yolov8m-p2_coco.pt")
if not os.path.exists(P2_INIT):
    import torch, ultralytics
    from ultralytics import YOLO
    os.makedirs(os.path.dirname(P2_INIT), exist_ok=True)
    os.makedirs(os.path.join(CFG["local_root"], "weights"), exist_ok=True)
    os.chdir(os.path.join(CFG["local_root"], "weights")) # pretrained .pt files are downloaded here (as in train_ultralytics)
    m = YOLO("yolov8m-p2.yaml").load("yolov8m.pt") # prints "Transferred N/M items from pretrained weights"
    torch.save({"model": copy.deepcopy(m.model).half(), "train_args": {}, "date": ml.now(), "version": ultralytics.__version__}, P2_INIT)
    del m
print("P2 init:", P2_INIT)

Transferred 319/581 items from pretrained weights
P2 init: /content/drive/MyDrive/MAVREC/Final_Project_Results/p2_init/yolov8m-p2_coco.pt


## 2 · Final model (reference)
Same spec as row D of notebook 08. It is already trained and evaluated, so this only loads it (in smoke mode it trains the tiny smoke version).

In [3]:
BEST_RES = 1280
FINAL_SPEC = ml.make_spec("yolov8m", f"aerial_train_{N_LARGE}", BEST_RES, stage="S3")
FINAL = FINAL_SPEC["experiment_id"]
if not CFG.get("smoke"):
    assert FINAL == "S3_yolov8m_n2000_r1280_full_COCO_s42", FINAL
ml.run_experiment(FINAL_SPEC, eval_splits=("val", "test")) # skips: already done
print(FINAL, "->", ml.load_record(FINAL)["checkpoint_path"])

[S3_yolov8m_n2000_r1280_full_COCO_s42] done — skipping (force=True to redo)
S3_yolov8m_n2000_r1280_full_COCO_s42 -> /content/drive/MyDrive/MAVREC/Final_Project_Results/runs/S3_yolov8m_n2000_r1280_full_COCO_s42/train/weights/last.pt


## 3 · Test-time augmentation (no training)
Each variant gets its own record (`S7_..._tta`, `S7_..._tta_up`) that points at the final model's weights, so it appears as a separate row in the results without touching the final model's record. FPS is measured with TTA on, so the speed cost is visible.

In [ ]:
class BuiltinTTAPredictor(ml.UltralyticsPredictor):
    """Ultralytics TTA (augment=True): flip + scales 1.0 / 0.83 / 0.67, merged inside the model."""
    def predict(self, images, res):
        bgr = [np.ascontiguousarray(im[:, :, ::-1]) for im in images]
        rs = self.model.predict(bgr, imgsz=int(res), conf=self.conf, iou=self.iou, max_det=self.max_det, augment=True,
                                quantize=16 if self.half else None, device=self.device, verbose=False)
        return [(r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy(), r.boxes.cls.cpu().numpy().astype(int)) for r in rs]


class FlipUpscaleTTAPredictor(ml.UltralyticsPredictor):
    """Flip x up-scale TTA: every (scale, flip) pass in native box coordinates, merged with class-wise NMS."""
    SCALES = (1.0, 1.25)

    def predict(self, images, res):
        import torch, torchvision
        parts = [[] for _ in images]
        for s in self.SCALES:
            size = int(round(res * s / 32) * 32)
            for flip in (False, True):
                ims = [im[:, ::-1] for im in images] if flip else images
                for k, (b, sc, l) in enumerate(super().predict(ims, size)):
                    if flip:
                        w = images[k].shape[1]
                        b = b.copy()
                        b[:, [0, 2]] = w - b[:, [2, 0]]
                    parts[k].append((b, sc, l))
        out = []
        for p in parts:
            b, sc, l = (np.concatenate([q[i] for q in p]) for i in range(3))
            if len(b):
                keep = torchvision.ops.batched_nms(torch.from_numpy(b).float(), torch.from_numpy(sc).float(),
                                                   torch.from_numpy(l).long(), self.iou).numpy()[: self.max_det]
                b, sc, l = b[keep], sc[keep], l[keep]
            out.append((b, sc, l))
        return out


def tta_record(src_id, tag, notes):
    """A record for an inference-only variant of `src_id`: same weights / training info, stage S7, own evals."""
    new_id = src_id.replace("S3_", "S7_", 1) + "_" + tag
    if ml.load_record(new_id) is None:
        r = copy.deepcopy(ml.load_record(src_id))
        r.update(experiment_id=new_id, stage="S7", notes=notes, evals={}, status="done", created=ml.now())
        r["spec"] = dict(r["spec"], experiment_id=new_id, stage="S7", notes=notes, tag=tag)
        ml.save_record(r)
    return new_id


W = ml.load_record(FINAL)["checkpoint_path"]
TTA = {
    "tta": (BuiltinTTAPredictor, "TTA: Ultralytics augment=True (flip + scales 1/0.83/0.67), final-model weights"),
    "tta_up": (FlipUpscaleTTAPredictor, "TTA: flip x scales 1/1.25 (1280+1600), class-wise NMS, final-model weights"),
}
TTA_IDS = {}
for tag, (cls, notes) in TTA.items():
    TTA_IDS[tag] = tta_record(FINAL, tag, notes)
    pred = cls(W, "yolov8m")
    for split in ("val", "test"):
        ml.evaluate(TTA_IDS[tag], split=split, mode="full", predictor=pred)
    del pred
    ml.free_gpu()

Pack 'aerial_1280' unpacked to local disk in 38s
  predicted 538 images (full) in 50s, 99431 detections
[S7_yolov8m_n2000_r1280_full_COCO_s42_tta] val|full|r1280: mAP50-95=70.294 AP50=91.922 APS=42.339 APt=18.316 R=89.883 FPS=17.2
  predicted 1614 images (full) in 133s, 288305 detections


## 4 · P2 small-object head (one training run, resumable)
Identical to the final model except the architecture: same 2000 frames, 1280, epochs, seed and evaluation. If the Colab session drops, re-run sections 0, 1 and this cell — training resumes from `last.pt`.

In [ ]:
P2_SPEC = ml.make_spec(P2, f"aerial_train_{N_LARGE}", BEST_RES, stage="S7", init=P2_INIT,
                       notes="YOLOv8m + P2 (stride-4) detection head, COCO-initialised; otherwise = final model")
print(P2_SPEC["experiment_id"], "| epochs", P2_SPEC["epochs"], "| batch", P2_SPEC["batch_size"])
ml.run_experiment(P2_SPEC, eval_splits=("val", "test"))

## 5 · Comparison with the final model
Δ columns are differences from the final model on the same split. APt = tiny objects, APS = small objects.

In [ ]:
ALL = ml.export_master_csv()
IDS = [FINAL, *TTA_IDS.values(), P2_SPEC["experiment_id"]]
COLS = ["mAP50_95", "mAP50", "APS", "APt", "APM", "recall", "F1", "FPS"]
T = ALL[ALL.experiment_id.isin(IDS) & ALL.eval_split.isin(["val", "test"]) & (ALL.inference_mode == "full")]
T = T[["experiment_id", "eval_split", *COLS, "train_time_h"]].copy()
base = T[T.experiment_id == FINAL].set_index("eval_split")[COLS]
for c in ["mAP50_95", "APS", "APt", "FPS"]:
    T[f"Δ {c}"] = (T[c] - T.eval_split.map(base[c])).round(2)
T["experiment_id"] = pd.Categorical(T.experiment_id, IDS, ordered=True)
T = T.sort_values(["eval_split", "experiment_id"]).reset_index(drop=True)
display(T)
out = os.path.join(CFG["tables_dir"] if "tables_dir" in CFG else os.path.join(DRIVE_ROOT, "Final_Project_Results", "tables"), "R_tta_p2.csv")
T.to_csv(out, index=False); print("saved", out)